In [3]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Define the Base Model
rf = RandomForestClassifier(random_state=42)

random_param_space = {
    'n_estimators': np.arange(10, 500, 50),
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': [2, 5, 10],
    'bootstrap': [True, False]
}

random_search = RandomizedSearchCV(
    estimator=rf, 
    param_distributions=random_param_space, 
    n_iter=10,      # Try 10 random combinations
    cv=5,           # 5-fold Cross-Validation
    verbose=1, 
    n_jobs=-1       #Use every single core available on this machine.
)

random_search.fit(X_train, y_train)
print(f"Best parameters from Random Search: {random_search.best_params_}")


grid_param_space = {
    'n_estimators': [80, 100, 120],
    'max_depth': [random_search.best_params_['max_depth']], # Use what we found
    'min_samples_split': [2, 3]
}

model1 = random_search.best_estimator_
accuracy1 = model1.score(X_test, y_test)

print(f"\nFinal Best Parameters: {random_search.best_params_}")
print(f"Accuracy on unseen test data: {accuracy1:.4f}")

grid_search = GridSearchCV(
    estimator=rf, 
    param_grid=grid_param_space, 
    cv=5,           # 5-fold Cross-Validation
    verbose=1, 
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

model2 = grid_search.best_estimator_
accuracy2 = model2.score(X_test, y_test)

print(f"\nFinal Best Parameters: {grid_search.best_params_}")
print(f"Accuracy on unseen test data: {accuracy2:.4f}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits


Best parameters from Random Search: {'n_estimators': np.int64(260), 'min_samples_split': 2, 'max_depth': 30, 'bootstrap': True}

Final Best Parameters: {'n_estimators': np.int64(260), 'min_samples_split': 2, 'max_depth': 30, 'bootstrap': True}
Accuracy on unseen test data: 0.9649
Fitting 5 folds for each of 6 candidates, totalling 30 fits

Final Best Parameters: {'max_depth': 30, 'min_samples_split': 3, 'n_estimators': 100}
Accuracy on unseen test data: 0.9649
